# DEH 30-Day PySpark Challenge
## Day 19 — JSON: Reading, explode & Nested Data

**Instructions:**
1. Click `File → Save a copy in Drive` before editing
2. Run each cell in order using `Shift + Enter`
3. Complete the assignment cells at the bottom

---

In [ ]:
!pip install pyspark --quiet

In [ ]:
import os
os.environ['AWS_ACCESS_KEY_ID']     = 'your-access-key-here'
os.environ['AWS_SECRET_ACCESS_KEY'] = 'your-secret-key-here'
os.environ['AWS_DEFAULT_REGION']    = 'us-east-1'
BUCKET = 'deh-pyspark-challenge-your-account-id'
print('Credentials set.')

In [ ]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType, ArrayType

spark = (
    SparkSession.builder
    .appName("DEH-PySpark-Challenge")
    .config("spark.jars.packages",
            "org.apache.hadoop:hadoop-aws:3.4.0,com.amazonaws:aws-java-sdk-bundle:1.12.367")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "com.amazonaws.auth.EnvironmentVariableCredentialsProvider")
    .getOrCreate()
)
print(f"Spark version: {spark.version}")

In [ ]:
orders_schema = StructType([
    StructField("order_id",       StringType(),  True),
    StructField("customer_id",    StringType(),  True),
    StructField("product_id",     StringType(),  True),
    StructField("order_date",     DateType(),    True),
    StructField("quantity",       IntegerType(), True),
    StructField("unit_price",     DoubleType(),  True),
    StructField("discount_pct",   IntegerType(), True),
    StructField("status",         StringType(),  True),
    StructField("payment_method", StringType(),  True),
    StructField("region",         StringType(),  True)
])

orders_df = spark.read.option("header","true").schema(orders_schema).csv(f"s3a://{BUCKET}/data/orders.csv")
print(f"Orders: {orders_df.count()}")

## Step 5 — Reading JSON from S3 with nested structure

In [ ]:
# Read orders.json from S3 — schema with nested struct and array inferred automatically
orders_json_df = spark.read.json(f"s3a://{BUCKET}/data/orders.json")

orders_json_df.printSchema()
orders_json_df.show(3, truncate=False)

## Step 6 — Accessing nested fields with dot notation

In [ ]:
# Access nested customer fields directly using dot notation
orders_json_df.select(
    F.col("order_id"),
    F.col("customer.id").alias("customer_id"),
    F.col("customer.name").alias("customer_name"),
    F.col("customer.city").alias("city"),
    F.col("customer.segment").alias("segment"),
    F.col("amount")
).show(5, truncate=False)

## Step 7 — from_json() — parsing a JSON string column

Use this when JSON arrives nested inside a string column (e.g. a Kafka message value).

In [ ]:
# A column containing a raw JSON string
raw_df = spark.createDataFrame(
    [('{"id": "C001", "name": "James Anderson", "city": "New York"}',)],
    ["customer_json"]
)

customer_schema = StructType([
    StructField("id",   StringType(), True),
    StructField("name", StringType(), True),
    StructField("city", StringType(), True)
])

raw_df.withColumn("customer", F.from_json(F.col("customer_json"), customer_schema)) \
      .select("customer.id", "customer.name", "customer.city") \
      .show()

## Step 8 — Handling malformed JSON

In [ ]:
# PERMISSIVE is the default — Spark automatically captures broken JSON in _corrupt_record
dirty_json_df = spark.read \
    .option("mode", "PERMISSIVE") \
    .json(f"s3a://{BUCKET}/data/orders_dirty.json")

dirty_json_df.printSchema()
print(f"Total rows: {dirty_json_df.count()}")

# Rows Spark could not parse at all
dirty_json_df.filter(F.col("_corrupt_record").isNotNull()) \
    .select("_corrupt_record") \
    .show(truncate=False)

**Key difference from CSV:** Spark adds `_corrupt_record` automatically for JSON — no need to add it to the schema manually. But type mismatches inside otherwise-valid JSON (like `amount` being a string) are silently nulled, not flagged as corrupt. Always check for unexpected nulls too.

## Step 9 — explode() on real data: orders.json items array

In [ ]:
# orders.json has an "items" array — one or more product_ids per order
orders_json_df.select("order_id", "items").show(5, truncate=False)

# explode — one row per item
orders_json_df.withColumn("product_id", F.explode(F.col("items"))) \
    .select("order_id", "product_id") \
    .show(10)

## Step 10 — explode() on inline data — a second example

In [ ]:
data = [
    ("C001", "James",  ["O0001", "O0021", "O0046"]),
    ("C002", "Maria",  ["O0002", "O0022", "O0047"]),
    ("C003", "Robert", ["O0003"])
]

df = spark.createDataFrame(data, ["customer_id", "name", "order_ids"])
print("Before explode:")
df.show()

print("After explode:")
df.withColumn("order_id", F.explode(F.col("order_ids"))) \
  .select("customer_id", "name", "order_id") \
  .show()

## Step 11 — explode_outer() — keeping empty/null arrays

In [ ]:
data2 = [
    ("C001", "James",   ["O0001", "O0021"]),
    ("C004", "Linda",   []),
    ("C005", "Michael", None)
]

df2 = spark.createDataFrame(data2, ["customer_id", "name", "order_ids"])

print("explode() — drops empty/null arrays:")
df2.withColumn("order_id", F.explode(F.col("order_ids"))).show()

print("explode_outer() — keeps empty/null arrays as null:")
df2.withColumn("order_id", F.explode_outer(F.col("order_ids"))).show()

## Step 12 — collect_list() and collect_set()

In [ ]:
orders_df.groupBy("customer_id").agg(
    F.collect_list("order_id").alias("all_orders"),
    F.collect_set("status").alias("unique_statuses"),
    F.count("order_id").alias("order_count")
).orderBy("customer_id").show(5, truncate=False)

---
## Assignment — Day 19

---

### Task 1
Create a DataFrame from JSON strings where each row has a nested `address` object with `street`, `city`, and `zip`.  
Use `from_json()` and dot notation to extract each field into its own column.

In [ ]:
# Task 1 — Write your code here


### Task 2
Create a DataFrame where each customer has an array of product categories they've purchased.  
Use `explode()` to create one row per category.  
How many total rows does the exploded DataFrame have?

In [ ]:
# Task 2 — Write your code here


### Task 3
Using `orders.csv`, use `collect_list()` to group all `order_id` values per `customer_id`.  
Then use `collect_set()` to get the unique statuses per customer.  
Show both alongside the order count.

In [ ]:
# Task 3 — Write your code here


### Task 4
Create a DataFrame with a mix of customers — some with order arrays, some with empty arrays, some with null.  
Use both `explode()` and `explode_outer()` and compare the row counts.  
Which one should you use in production and why?

In [ ]:
# Task 4 — Write your code here


---
*DEH 30-Day PySpark Challenge · Data Engineering Hub · RADE Program*